In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms.v2 import RandomHorizontalFlip, RandomVerticalFlip, RandomRotation
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, Dataset, DataLoader
import torch.optim as optim
import torch


dataset_path = 'data/train'
dataset = ImageFolder(dataset_path)

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

class TransformDataset(Dataset):
  def __init__(self, dataset, transforms):
    super(TransformDataset, self).__init__()
    self.dataset = dataset
    self.transforms = transforms

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    x, y = self.dataset[idx]
    return self.transforms(x), y

test_transforms = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize((0.5), (0.5))
]) 

train_transforms = Compose([
RandomHorizontalFlip(p=0.2),
RandomVerticalFlip(p=0.2),
RandomRotation([-5, 5], fill=255.) # фон изображений белый, поэтому заполнение белым 
])

# Визуализация датасета до трансформаций
fig = plt.figure(figsize=(10,5))
for index in range (1, 11):
    image, label = train_dataset[index]
    print(dataset.classes[label])
    plt.subplot(1, 10, index)
    plt.imshow(image)
    plt.axis('off')

train_dataset = TransformDataset(train_dataset, train_transforms)
val_dataset = TransformDataset(val_dataset, test_transforms) 

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Проверка
print("Количество изображений в train:", len(train_dataset))
print("Количество изображений в val:", len(val_dataset))
print("Список классов:", dataset.classes) 


In [16]:
from torchsummary import summary
from torchvision.models import mobilenet_v3_small
import torch.nn as nn

# Загрузка модели
model = mobilenet_v3_small(weights='IMAGENET1K_V1')

# Замена слоя для классификации
model.classifier = nn.Linear(in_features=576, out_features=12, bias=True)

# Заморозка слоёв
for param in model.parameters():
    param.requires_grad = False

# Разморозка полносвязного слоя classifier
for param in model.classifier.parameters():
    param.requires_grad = True
    
# Проверка
summary(model, input_size=(3, 224, 224), device='cpu') 

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 112, 112]             432
       BatchNorm2d-2         [-1, 16, 112, 112]              32
         Hardswish-3         [-1, 16, 112, 112]               0
            Conv2d-4           [-1, 16, 56, 56]             144
       BatchNorm2d-5           [-1, 16, 56, 56]              32
              ReLU-6           [-1, 16, 56, 56]               0
 AdaptiveAvgPool2d-7             [-1, 16, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             136
              ReLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 16, 1, 1]             144
      Hardsigmoid-11             [-1, 16, 1, 1]               0
SqueezeExcitation-12           [-1, 16, 56, 56]               0
           Conv2d-13           [-1, 16, 56, 56]             256
      BatchNorm2d-14           [-1, 16,

In [19]:
# Вставьте код для загрузки датасета из прошлого задания
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms.v2 import RandomHorizontalFlip, RandomVerticalFlip, RandomRotation
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, Dataset, DataLoader
import torch.optim as optim
import torch

from torchvision.models import mobilenet_v3_small
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v3_small
import torch.nn as nn

# Загрузка модели
model = mobilenet_v3_small(weights='IMAGENET1K_V1')

# Замена слоя для классификации
model.classifier = nn.Linear(in_features=576, out_features=12, bias=True)

# Заморозка слоёв
for param in model.parameters():
    param.requires_grad = False

# Разморозка полносвязного слоя classifier
for param in model.classifier.parameters():
    param.requires_grad = True
    


dataset_path = 'data/train'
dataset = ImageFolder(dataset_path)

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

class TransformDataset(Dataset):
  def __init__(self, dataset, transforms):
    super(TransformDataset, self).__init__()
    self.dataset = dataset
    self.transforms = transforms

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    x, y = self.dataset[idx]
    return self.transforms(x), y

test_transforms = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize((0.5,), (0.5,))
]) 

train_transforms = Compose([
    Resize((224, 224)),
    RandomHorizontalFlip(p=0.2),
    RandomVerticalFlip(p=0.2),
    RandomRotation([-5, 5], fill=255),
    ToTensor(),
    Normalize((0.5,), (0.5,)) 
])

train_dataset = TransformDataset(train_dataset, train_transforms)
val_dataset = TransformDataset(val_dataset, test_transforms) 

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

   
# Настройка гиперпараметров
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 10
best_vloss = 1e5

# Код обучения из прошлого урока
def train_one_epoch(epoch_index):
    running_loss = 0.
    last_loss = 0.

    for batch_index, data in enumerate(train_loader):
        # Извлечение батча
        inputs, labels = data
        # Обнуление градиентов
        optimizer.zero_grad()
        # Прямое распространение
        outputs = model(inputs)
        # Подсчёт ошибки
        loss = criterion(outputs, labels)
        # Обратное распространение
        loss.backward()
        # Обновление весов
        optimizer.step()

        # Суммирование ошибки за последние 20 батчей
        running_loss += loss.item()
        if batch_index % 20 == 19:
            last_loss = running_loss / 20. # средняя ошибка за 20 батчей
            print(f'Эпоха: {epoch_index}, батч: {batch_index}, ошибка {last_loss}')
            running_loss = 0.

    return last_loss

for epoch in range(EPOCHS):
    print(f'Эпоха {epoch}')

    # Перевод модели в режим обучения
    model.train(True)
    # Эпоха обучения
    avg_loss = train_one_epoch(epoch)

    # Перевод модели в режим валидации
    model.eval()
    running_vloss = 0.0

    # Валидация
    with torch.no_grad():
        for i, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            voutputs = model(vinputs)
            vloss = criterion(voutputs, vlabels)
            running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)

    # Сохранение лучшей модели
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = f'color_classifier_{epoch}.pt'
        torch.save(model.state_dict(), model_path)

    print(f'В конце эпохи ошибка train {avg_loss}, ошибка val {avg_vloss}')

Эпоха 0
Эпоха: 0, батч: 19, ошибка 2.32460697889328
Эпоха: 0, батч: 39, ошибка 1.9945313215255738
Эпоха: 0, батч: 59, ошибка 1.7770867049694061
Эпоха: 0, батч: 79, ошибка 1.6461513042449951
Эпоха: 0, батч: 99, ошибка 1.5460712015628815
Эпоха: 0, батч: 119, ошибка 1.4393559098243713
Эпоха: 0, батч: 139, ошибка 1.3900915205478668
В конце эпохи ошибка train 1.3900915205478668, ошибка val 1.3733701705932617
Эпоха 1
Эпоха: 1, батч: 19, ошибка 1.2646795451641082
Эпоха: 1, батч: 39, ошибка 1.243239763379097
Эпоха: 1, батч: 59, ошибка 1.2469066679477692
Эпоха: 1, батч: 79, ошибка 1.217458075284958
Эпоха: 1, батч: 99, ошибка 1.1209157019853593
Эпоха: 1, батч: 119, ошибка 1.1271110713481902
Эпоха: 1, батч: 139, ошибка 1.1129161655902862
В конце эпохи ошибка train 1.1129161655902862, ошибка val 1.057471513748169
Эпоха 2
Эпоха: 2, батч: 19, ошибка 1.0827941030263901
Эпоха: 2, батч: 39, ошибка 1.0279394567012787
Эпоха: 2, батч: 59, ошибка 1.028295859694481
Эпоха: 2, батч: 79, ошибка 1.0634086489677